In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
import numpy as np
import matplotlib.pyplot as plt
import os
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# ==================== 设备配置 ====================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)
np.random.seed(42)
print(f"使用设备: {device}")

# ==================== 几何参数 ====================
d0 = 0.05  # 内管直径（m）
d1 = 0.13  # 外壳直径（m）
r0 = d0 / 2  # 内管半径
r1 = d1 / 2  # 外壳半径
L_char = d1 - d0  # 特征长度：环形域宽度 (m)

# ==================== 材料参数 ====================
# PCM（石蜡）- 来自论文Table 1
rho_s = 880.0      # 固相密度 (kg/m³)
rho_l = 760.0      # 液相密度 (kg/m³)
cp_s = 2180.0      # 固相定压比热容 (J/(kg·K))
cp_l = 2390.0      # 液相定压比热容 (J/(kg·K))
lambda_s = 0.4     # 固相导热系数 (W/(m·K))
lambda_l = 0.15    # 液相导热系数 (W/(m·K))
mu_l = 0.001       # 液相粘度 (kg/(m·s))
L = 255000.0       # 相变潜热 (J/kg)
Tpc = 316.15       # 相变温度 (K)
DeltaT = 6.0       # 相变温度区间 (K)
alpha = 1.0e-4     # 体膨胀系数 (1/K)
g = 9.81           # 重力加速度 (m/s²)

# 高导热材料（铜）
rho_Cu = 8960.0    # 密度 (kg/m³)
lambda_Cu = 400.0  # 导热系数 (W/(m·K))
cp_Cu = 385.0      # 定压比热容 (J/(kg·K))

# ==================== 过程参数 ====================
T_initial = 290.0  # 初始温度 (K)
T_heating = 360.0  # 加热温度 (K)
T_cooling = 290.0  # 冷却温度 (K)

# ==================== 拓扑优化参数 ====================
phi_total = 0.3  # 高导热材料体积比约束
case = 1         # 优化目标选择：1=平均温度，2=温度均方差，3=多目标

# ==================== 数值稳定性参数 ====================
eps = 1e-8      # 防止除零的小量

# ==================== 简化的物理场网络 ====================
class SimplePhysicsNN(nn.Module):
    """简化的物理场网络 - 专注于温度场"""
    
    def __init__(self, input_dim=3, hidden_dim=64, num_layers=4):
        super(SimplePhysicsNN, self).__init__()
        
        # 构建网络
        layers = []
        layers.append(nn.Linear(input_dim, hidden_dim))
        layers.append(nn.Tanh())
        
        for _ in range(num_layers - 2):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(nn.Tanh())
        
        layers.append(nn.Linear(hidden_dim, 1))  # 只输出温度
        
        self.net = nn.Sequential(*layers)
        
        # 温度参数
        self.register_buffer('T_initial', torch.tensor(T_initial))
        self.register_buffer('T_heating', torch.tensor(T_heating))
        self.register_buffer('T_cooling', torch.tensor(T_cooling))
        self.register_buffer('Tpc', torch.tensor(Tpc))
        
        # 初始化权重
        self._initialize_weights()
    
    def _initialize_weights(self):
        """初始化网络权重"""
        for layer in self.net:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_uniform_(layer.weight, gain=0.1)
                nn.init.zeros_(layer.bias)
    
    def forward(self, x):
        """
        前向传播
        输入: x [batch, 3] - 无量纲坐标 (x*, y*, τ*)
        输出: T
        """
        # 网络预测
        out = self.net(x)
        
        # 温度在合理范围内 (290-360K)
        T = self.T_initial + (self.T_heating - self.T_initial) * torch.sigmoid(out)
        
        return T

# ==================== 简化的拓扑网络 ====================
class SimpleTopologyNN(nn.Module):
    """简化的拓扑网络 - 使用高斯滤波平滑"""
    
    def __init__(self, hidden_dim=64, num_layers=4):
        super(SimpleTopologyNN, self).__init__()
        
        # 构建网络
        layers = []
        layers.append(nn.Linear(2, hidden_dim))  # 只输入空间坐标
        layers.append(nn.Tanh())
        
        for _ in range(num_layers - 2):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(nn.Tanh())
        
        layers.append(nn.Linear(hidden_dim, 1))
        layers.append(nn.Sigmoid())  # 输出在[0,1]之间
        
        self.net = nn.Sequential(*layers)
        
        # 初始化权重
        self._initialize_weights()
        
        # 高斯滤波参数
        self.filter_size = 5
        self.sigma = 1.0
        
    def _initialize_weights(self):
        """初始化权重"""
        for layer in self.net:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_uniform_(layer.weight, gain=0.5)
                nn.init.zeros_(layer.bias)
    
    def apply_gaussian_filter(self, rho, grid_size=32):
        """应用高斯滤波平滑拓扑结构"""
        # 将rho重塑为2D网格
        rho_2d = rho.view(grid_size, grid_size)
        
        # 创建高斯滤波器
        filter_range = torch.arange(-self.filter_size//2 + 1, self.filter_size//2 + 1)
        x = filter_range.unsqueeze(0).repeat(self.filter_size, 1)
        y = filter_range.unsqueeze(1).repeat(1, self.filter_size)
        gaussian = torch.exp(-(x**2 + y**2) / (2 * self.sigma**2))
        gaussian = gaussian / gaussian.sum()
        
        # 应用卷积（简化版本）
        rho_padded = torch.nn.functional.pad(rho_2d.unsqueeze(0).unsqueeze(0), 
                                           (self.filter_size//2, self.filter_size//2, 
                                            self.filter_size//2, self.filter_size//2), 
                                           mode='reflect')
        
        # 简单卷积
        rho_smoothed = torch.zeros_like(rho_2d)
        for i in range(grid_size):
            for j in range(grid_size):
                patch = rho_padded[0, 0, i:i+self.filter_size, j:j+self.filter_size]
                rho_smoothed[i, j] = (patch * gaussian).sum()
        
        return rho_smoothed.flatten().unsqueeze(1)
    
    def forward(self, x_space):
        """
        前向传播
        输入: x_space [batch, 2] - 空间坐标
        输出: rho [batch, 1] - 拓扑设计变量 ∈ [0,1]
        """
        # 基础网络预测
        rho = self.net(x_space)
        
        return rho

# ==================== 材料模型 ====================
class MaterialModel:
    """简化的材料模型"""
    
    def __init__(self):
        # PCM材料参数
        self.Tpc = Tpc
        self.DeltaT = DeltaT
        self.L = L
        self.rho_s = rho_s
        self.rho_l = rho_l
        self.cp_s = cp_s
        self.cp_l = cp_l
        self.lambda_s = lambda_s
        self.lambda_l = lambda_l
        
        # 铜的材料参数
        self.rho_Cu = rho_Cu
        self.lambda_Cu = lambda_Cu
        self.cp_Cu = cp_Cu
        
    def compute_liquid_fraction(self, T):
        """计算液相率"""
        if isinstance(T, torch.Tensor):
            # 使用平滑的过渡函数
            normalized = (T - self.Tpc) / (self.DeltaT / 2.0)
            phi = 0.5 * (1.0 + torch.tanh(normalized))
            return phi
        else:
            normalized = (T - self.Tpc) / (self.DeltaT / 2.0)
            phi = 0.5 * (1.0 + np.tanh(normalized))
            return np.clip(phi, 0.0, 1.0)
    
    def compute_effective_cp(self, T):
        """计算有效比热容（包含潜热）"""
        phi = self.compute_liquid_fraction(T)
        
        if isinstance(T, torch.Tensor):
            # 显热部分
            cp_sensible = self.cp_s + (self.cp_l - self.cp_s) * phi
            
            # 潜热部分 - 高斯分布
            sigma = self.DeltaT / 4.0
            exponent = -((T - self.Tpc) ** 2) / (2 * sigma ** 2 + eps)
            gaussian = torch.exp(exponent) / (sigma * np.sqrt(2 * np.pi) + eps)
            
            cp_latent = self.L * gaussian
            
            return cp_sensible + cp_latent, phi
        else:
            cp_sensible = self.cp_s + (self.cp_l - self.cp_s) * phi
            
            sigma = self.DeltaT / 4.0
            exponent = -((T - self.Tpc) ** 2) / (2 * sigma ** 2 + eps)
            gaussian = np.exp(exponent) / (sigma * np.sqrt(2 * np.pi) + eps)
            
            cp_latent = self.L * gaussian
            
            return cp_sensible + cp_latent, phi
    
    def compute_properties(self, T, rho_design):
        """计算混合材料属性"""
        # 计算PCM属性
        cp_eff, phi = self.compute_effective_cp(T)
        
        if isinstance(T, torch.Tensor):
            # 确保rho_design在[0,1]范围内
            rho_design = torch.clamp(rho_design, 0.0, 1.0)
            
            # SIMP方法
            p = 3.0  # SIMP惩罚因子
            rho_p = torch.pow(rho_design, p)
            
            # 混合密度
            rho_pcm = self.rho_s * (1 - phi) + self.rho_l * phi
            rho_total = rho_p * self.rho_Cu + (1 - rho_p) * rho_pcm
            
            # 混合导热系数
            lambda_pcm = self.lambda_s * (1 - phi) + self.lambda_l * phi
            lambda_total = rho_p * self.lambda_Cu + (1 - rho_p) * lambda_pcm
            
            # 混合比热容
            mass_Cu = rho_p * self.rho_Cu
            mass_pcm = (1 - rho_p) * rho_pcm
            mass_total = mass_Cu + mass_pcm + eps
            cp_total = (mass_Cu * self.cp_Cu + mass_pcm * cp_eff) / mass_total
            
            return rho_total, lambda_total, cp_total, phi, rho_design
        else:
            rho_design = np.clip(rho_design, 0.0, 1.0)
            p = 3.0
            rho_p = np.power(rho_design, p)
            
            rho_pcm = self.rho_s * (1 - phi) + self.rho_l * phi
            rho_total = rho_p * self.rho_Cu + (1 - rho_p) * rho_pcm
            
            lambda_pcm = self.lambda_s * (1 - phi) + self.lambda_l * phi
            lambda_total = rho_p * self.lambda_Cu + (1 - rho_p) * lambda_pcm
            
            mass_Cu = rho_p * self.rho_Cu
            mass_pcm = (1 - rho_p) * rho_pcm
            mass_total = mass_Cu + mass_pcm + eps
            cp_total = (mass_Cu * self.cp_Cu + mass_pcm * cp_eff) / mass_total
            
            return rho_total, lambda_total, cp_total, phi, rho_design

# ==================== 简化的PINN模型 ====================
class SimplePINN(nn.Module):
    """简化的PINN模型"""
    
    def __init__(self):
        super(SimplePINN, self).__init__()
        
        # 物理场网络
        self.physics_nn = SimplePhysicsNN(input_dim=3, hidden_dim=64, num_layers=4)
        
        # 拓扑网络
        self.topology_nn = SimpleTopologyNN(hidden_dim=64, num_layers=4)
        
        # 材料模型
        self.material = MaterialModel()
        
        # 过程标志
        self.is_heating = True
    
    def set_heating_mode(self, is_heating=True):
        """设置加热模式"""
        self.is_heating = is_heating
    
    def forward(self, x):
        """
        前向传播
        输入: x [batch, 3] - 坐标 (x, y, t)
        输出: T, rho
        """
        # 提取空间坐标和时间
        x_space = x[:, 0:2]
        
        # 物理场预测（温度）
        T = self.physics_nn(x)
        
        # 拓扑设计变量
        rho = self.topology_nn(x_space)
        
        return T, rho
    
    def compute_pde_residuals(self, x):
        """
        计算PDE残差 - 简化的热传导方程
        """
        # 确保输入需要梯度
        if not x.requires_grad:
            x = x.clone().requires_grad_(True)
        
        # 前向传播
        T, rho = self(x)
        
        # 计算材料属性
        rho_total, lambda_total, cp_total, phi, rho_design = self.material.compute_properties(T, rho)
        
        # 计算温度梯度
        grad_T = torch.autograd.grad(
            T, x, 
            grad_outputs=torch.ones_like(T),
            create_graph=True, 
            retain_graph=True,
            allow_unused=True
        )[0]
        if grad_T is None:
            grad_T = torch.zeros_like(x)
        
        # 转换为实际导数（简化处理）
        dTdx = grad_T[:, 0:1]
        dTdy = grad_T[:, 1:2]
        dTdt = grad_T[:, 2:3]
        
        # 计算二阶导数
        if x.requires_grad:
            # 计算dTdx的梯度
            grad_dTdx = torch.autograd.grad(
                dTdx, x,
                grad_outputs=torch.ones_like(dTdx),
                create_graph=True,
                retain_graph=True,
                allow_unused=True
            )[0]
            if grad_dTdx is None:
                d2Tdx2 = torch.zeros_like(dTdx)
            else:
                d2Tdx2 = grad_dTdx[:, 0:1]
            
            # 计算dTdy的梯度
            grad_dTdy = torch.autograd.grad(
                dTdy, x,
                grad_outputs=torch.ones_like(dTdy),
                create_graph=True,
                retain_graph=True,
                allow_unused=True
            )[0]
            if grad_dTdy is None:
                d2Tdy2 = torch.zeros_like(dTdy)
            else:
                d2Tdy2 = grad_dTdy[:, 1:2]
        else:
            d2Tdx2 = torch.zeros_like(dTdx)
            d2Tdy2 = torch.zeros_like(dTdy)
        
        # 简化的热传导方程：ρcp ∂T/∂t = λ ∇²T
        # 忽略对流项，专注于热传导
        
        # 热扩散系数
        alpha_total = lambda_total / (rho_total * cp_total + eps)
        
        # PDE残差
        energy_residual = dTdt - alpha_total * (d2Tdx2 + d2Tdy2)
        
        return {
            'energy': energy_residual,
            'T': T,
            'rho_design': rho_design,
            'phi': phi,
            'rho_total': rho_total,
            'lambda_total': lambda_total,
            'cp_total': cp_total
        }

# ==================== 采样器 ====================
class Sampler:
    """采样器"""
    
    def __init__(self, r0, r1):
        self.r0 = r0
        self.r1 = r1
        
    def sample_domain(self, N):
        """采样计算域内部点"""
        # 极坐标采样
        r = np.sqrt(np.random.uniform(r0**2, r1**2, N))
        theta = np.random.uniform(0, 2*np.pi, N)
        
        x = r * np.cos(theta)
        y = r * np.sin(theta)
        t = np.random.uniform(0, 1.0, N)  # 归一化时间
        
        points = np.column_stack([x, y, t])
        
        return torch.tensor(points, dtype=torch.float32)
    
    def sample_boundary(self, N, boundary='inner'):
        """采样边界点"""
        if boundary == 'inner':
            r = self.r0
        else:
            r = self.r1
        
        theta = np.random.uniform(0, 2*np.pi, N)
        x = r * np.cos(theta)
        y = r * np.sin(theta)
        t = np.random.uniform(0, 1.0, N)
        
        points = np.column_stack([x, y, t])
        
        return torch.tensor(points, dtype=torch.float32)
    
    def sample_initial(self, N):
        """采样初始条件点"""
        r = np.sqrt(np.random.uniform(r0**2, r1**2, N))
        theta = np.random.uniform(0, 2*np.pi, N)
        
        x = r * np.cos(theta)
        y = r * np.sin(theta)
        t = np.zeros(N)
        
        points = np.column_stack([x, y, t])
        
        return torch.tensor(points, dtype=torch.float32)
    
    def create_grid(self, N_r=30, N_theta=50, N_t=5):
        """创建均匀网格用于可视化"""
        r = np.linspace(r0, r1, N_r)
        theta = np.linspace(0, 2*np.pi, N_theta)
        t = np.linspace(0, 1.0, N_t)
        
        R, Theta, T = np.meshgrid(r, theta, t, indexing='ij')
        
        # 转换为直角坐标
        X = R * np.cos(Theta)
        Y = R * np.sin(Theta)
        
        points = np.column_stack([X.flatten(), Y.flatten(), T.flatten()])
        
        return torch.tensor(points, dtype=torch.float32), X, Y, T

# ==================== 损失计算器 ====================
class LossCalculator:
    """损失计算器"""
    
    def __init__(self, model):
        self.model = model
        self.sampler = Sampler(r0, r1)
        
        # 损失权重
        self.weights = {
            'pde': 1.0,
            'ic': 10.0,
            'bc_inner': 100.0,
            'bc_outer': 10.0,
            'volume': 5.0,
            'objective': 2.0,
            'smoothness': 1.0,  # 拓扑平滑性
        }
        
        # 当前体积分数
        self.current_volume = 0.0
        
        # 损失历史
        self.history = {key: [] for key in self.weights.keys()}
        self.history['total_loss'] = []
    
    def compute_total_loss(self):
        """计算总损失"""
        # 采样点
        N_domain = 500
        N_boundary = 100
        N_initial = 100
        
        x_domain = self.sampler.sample_domain(N_domain).to(device)
        x_inner = self.sampler.sample_boundary(N_boundary, 'inner').to(device)
        x_outer = self.sampler.sample_boundary(N_boundary, 'outer').to(device)
        x_initial = self.sampler.sample_initial(N_initial).to(device)
        
        # 计算各项损失
        losses = {}
        
        # PDE损失
        residuals = self.model.compute_pde_residuals(x_domain)
        losses['pde'] = torch.mean(residuals['energy'] ** 2)
        
        # 初始条件损失
        losses['ic'] = self.compute_ic_loss(x_initial)
        
        # 边界条件损失
        losses['bc_inner'] = self.compute_bc_loss(x_inner, 'inner')
        losses['bc_outer'] = self.compute_bc_loss(x_outer, 'outer')
        
        # 体积约束损失
        losses['volume'] = self.compute_volume_loss(x_domain)
        
        # 优化目标损失
        losses['objective'] = self.compute_objective_loss(residuals)
        
        # 拓扑平滑性损失
        losses['smoothness'] = self.compute_smoothness_loss(x_domain)
        
        # 记录历史
        for key in losses:
            self.history[key].append(losses[key].item())
        
        # 计算加权总损失
        total_loss = torch.tensor(0.0).to(device)
        for key, loss in losses.items():
            total_loss += self.weights[key] * loss
        
        self.history['total_loss'].append(total_loss.item())
        
        return total_loss, losses
    
    def compute_ic_loss(self, x):
        """计算初始条件损失"""
        T, rho = self.model(x)
        
        # 初始温度应为T_initial
        T_target = self.model.physics_nn.T_initial
        T_loss = torch.mean((T - T_target) ** 2)
        
        return T_loss
    
    def compute_bc_loss(self, x, boundary='inner'):
        """计算边界条件损失"""
        T, rho = self.model(x)
        
        if boundary == 'inner':
            # 内壁：恒温边界
            if self.model.is_heating:
                T_target = self.model.physics_nn.T_heating
            else:
                T_target = self.model.physics_nn.T_cooling
            
            T_loss = torch.mean((T - T_target) ** 2)
            return T_loss
        else:
            # 外壁：绝热边界 - 简化处理
            return torch.tensor(0.0).to(device)
    
    def compute_volume_loss(self, x):
        """计算体积约束损失"""
        x_space = x[:, 0:2]
        rho = self.model.topology_nn(x_space)
        
        # 计算平均体积分数
        volume = torch.mean(rho)
        self.current_volume = volume.item()
        
        # 体积约束损失
        violation = (volume - phi_total) ** 2
        return violation
    
    def compute_objective_loss(self, residuals):
        """计算优化目标损失"""
        T = residuals['T']
        
        if case == 1:
            # 最小化平均温度
            T_avg = torch.mean(T)
            # 目标是使温度接近相变温度
            target_temp = Tpc + 2.0
            return (T_avg - target_temp) ** 2
        elif case == 2:
            # 最小化温度方差
            T_avg = torch.mean(T)
            T_var = torch.mean((T - T_avg) ** 2)
            return T_var
        else:
            # 多目标优化
            T_avg = torch.mean(T)
            T_var = torch.mean((T - T_avg) ** 2)
            return 0.5 * (T_avg - 320.0) ** 2 + 0.5 * T_var
    
    def compute_smoothness_loss(self, x):
        """计算拓扑平滑性损失"""
        x_space = x[:, 0:2]
        rho = self.model.topology_nn(x_space)
        
        # 计算梯度（鼓励平滑）
        if x_space.requires_grad:
            grad_rho = torch.autograd.grad(
                rho, x_space,
                grad_outputs=torch.ones_like(rho),
                create_graph=True,
                retain_graph=True,
                allow_unused=True
            )[0]
            if grad_rho is None:
                return torch.tensor(0.0).to(device)
            
            smoothness_loss = torch.mean(grad_rho ** 2)
            return smoothness_loss
        else:
            return torch.tensor(0.0).to(device)

# ==================== 训练器 ====================
class Trainer:
    """训练器"""
    
    def __init__(self, model):
        self.model = model
        self.loss_calculator = LossCalculator(model)
        
        # 训练阶段
        self.stages = [
            {'name': 'pretrain', 'epochs': 100, 'lr': 1e-3, 'desc': '预训练'},
            {'name': 'refine', 'epochs': 100, 'lr': 5e-4, 'desc': '精细化训练'},
        ]
        
        # 训练历史
        self.history = {
            'total_loss': [],
            'volume': [],
            'stage': [],
            'learning_rate': []
        }
        
        # 创建保存目录
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        self.save_dir = f"./results_simple/case{case}_{timestamp}"
        os.makedirs(self.save_dir, exist_ok=True)
        
        # 最佳模型跟踪
        self.best_loss = float('inf')
        self.best_model_state = None
    
    def train(self):
        """训练模型"""
        print("="*60)
        print("开始训练简化版拓扑优化PINN")
        print("="*60)
        
        total_epochs = 0
        
        for stage_idx, stage_config in enumerate(self.stages):
            stage_name = stage_config['name']
            epochs = stage_config['epochs']
            lr = stage_config['lr']
            desc = stage_config['desc']
            
            print(f"\n{'='*60}")
            print(f"阶段 {stage_idx+1}/{len(self.stages)}: {desc}")
            print(f"轮次: {epochs}, 学习率: {lr}")
            print(f"{'='*60}")
            
            # 设置优化器
            params = list(self.model.parameters())
            optimizer = optim.Adam(params, lr=lr)
            scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=lr/10)
            
            # 阶段训练
            for epoch in range(epochs):
                total_epochs += 1
                
                try:
                    # 计算损失
                    total_loss, losses = self.loss_calculator.compute_total_loss()
                    
                    # 检查NaN
                    if torch.isnan(total_loss) or torch.isinf(total_loss):
                        print(f"警告: 第{epoch+1}轮损失为NaN或inf，跳过本轮")
                        continue
                    
                    # 反向传播
                    optimizer.zero_grad()
                    total_loss.backward()
                    
                    # 梯度裁剪
                    torch.nn.utils.clip_grad_norm_(params, 1.0)
                    
                    # 优化
                    optimizer.step()
                    scheduler.step()
                    
                    # 记录历史
                    self.history['total_loss'].append(total_loss.item())
                    self.history['volume'].append(self.loss_calculator.current_volume)
                    self.history['stage'].append(stage_idx)
                    self.history['learning_rate'].append(optimizer.param_groups[0]['lr'])
                    
                    # 保存最佳模型
                    if total_loss.item() < self.best_loss:
                        self.best_loss = total_loss.item()
                        self.best_model_state = self.model.state_dict().copy()
                    
                    # 输出进度
                    if (epoch + 1) % 10 == 0:
                        print(f"Epoch {total_epochs:4d} | Loss: {total_loss.item():.4e} | "
                              f"Volume: {self.loss_calculator.current_volume:.4f} | "
                              f"PDE: {losses['pde'].item():.2e} | "
                              f"BC: {losses['bc_inner'].item():.2e} | "
                              f"LR: {optimizer.param_groups[0]['lr']:.2e}")
                    
                    # 保存检查点
                    if (epoch + 1) % 50 == 0:
                        self.save_checkpoint(f"{stage_name}_epoch{epoch+1}")
                
                except Exception as e:
                    print(f"训练出错: {e}")
                    import traceback
                    traceback.print_exc()
                    # 重置梯度，继续训练
                    optimizer.zero_grad()
                    continue
            
            # 保存阶段检查点
            self.save_checkpoint(stage_name)
        
        print("\n训练完成!")
        
        # 加载最佳模型
        if self.best_model_state is not None:
            self.model.load_state_dict(self.best_model_state)
            print(f"加载最佳模型 (损失: {self.best_loss:.4e})")
        
        self.save_results()
    
    def save_checkpoint(self, checkpoint_name):
        """保存检查点"""
        checkpoint = {
            'model_state_dict': self.model.state_dict(),
            'history': self.history,
            'loss_history': self.loss_calculator.history,
            'best_loss': self.best_loss
        }
        
        filename = f"{self.save_dir}/checkpoint_{checkpoint_name}.pth"
        torch.save(checkpoint, filename)
    
    def save_results(self):
        """保存结果"""
        # 保存最终模型
        torch.save(self.model.state_dict(), f"{self.save_dir}/model_final.pth")
        
        # 保存最佳模型
        if self.best_model_state is not None:
            torch.save(self.best_model_state, f"{self.save_dir}/model_best.pth")
        
        # 保存训练历史
        np.savez(f"{self.save_dir}/training_history.npz",
                 total_loss=self.history['total_loss'],
                 volume=self.history['volume'],
                 stage=self.history['stage'],
                 learning_rate=self.history['learning_rate'])
        
        # 保存损失历史
        for key in self.loss_calculator.history:
            np.save(f"{self.save_dir}/loss_{key}.npy", np.array(self.loss_calculator.history[key]))
        
        # 绘制训练曲线
        self.plot_training_history()
        
        print(f"\n所有结果已保存到: {self.save_dir}")
    
    def plot_training_history(self):
        """绘制训练历史曲线"""
        if not self.history['total_loss']:
            print("警告: 没有训练历史数据")
            return
        
        fig, axes = plt.subplots(2, 2, figsize=(12, 10))
        
        # 总损失
        axes[0, 0].semilogy(self.history['total_loss'])
        axes[0, 0].set_xlabel('Epoch')
        axes[0, 0].set_ylabel('Total Loss')
        axes[0, 0].set_title('Total Loss History')
        axes[0, 0].grid(True, alpha=0.3)
        
        # 体积分数
        axes[0, 1].plot(self.history['volume'])
        axes[0, 1].axhline(y=phi_total, color='r', linestyle='--', label=f'Target: {phi_total}')
        axes[0, 1].set_xlabel('Epoch')
        axes[0, 1].set_ylabel('Volume Fraction')
        axes[0, 1].set_title('Volume Constraint')
        axes[0, 1].legend()
        axes[0, 1].grid(True, alpha=0.3)
        
        # 损失分量
        loss_keys = ['pde', 'bc_inner', 'ic', 'volume', 'objective']
        colors = plt.cm.Set1(np.linspace(0, 1, len(loss_keys)))
        for i, key in enumerate(loss_keys):
            if key in self.loss_calculator.history and self.loss_calculator.history[key]:
                axes[1, 0].semilogy(self.loss_calculator.history[key], 
                                  label=key, color=colors[i], alpha=0.7)
        axes[1, 0].set_xlabel('Epoch')
        axes[1, 0].set_ylabel('Loss')
        axes[1, 0].set_title('Loss Components')
        axes[1, 0].legend()
        axes[1, 0].grid(True, alpha=0.3)
        
        # 学习率
        axes[1, 1].semilogy(self.history['learning_rate'])
        axes[1, 1].set_xlabel('Epoch')
        axes[1, 1].set_ylabel('Learning Rate')
        axes[1, 1].set_title('Learning Rate Schedule')
        axes[1, 1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(f"{self.save_dir}/training_history.png", dpi=300)
        plt.close()

# ==================== 验证函数 ====================
def validate_model(model):
    """验证模型"""
    print("\n" + "="*60)
    print("模型验证")
    print("="*60)
    
    sampler = Sampler(r0, r1)
    
    # 采样验证点
    x_val = sampler.sample_domain(200).to(device)
    x_inner = sampler.sample_boundary(50, 'inner').to(device)
    x_initial = sampler.sample_initial(50).to(device)
    
    with torch.no_grad():
        # 验证边界条件
        T_inner, rho_inner = model(x_inner)
        
        if model.is_heating:
            T_target = model.physics_nn.T_heating
        else:
            T_target = model.physics_nn.T_cooling
        
        T_error_inner = torch.mean(torch.abs(T_inner - T_target)).item()
        print(f"内壁温度误差: {T_error_inner:.4f} K")
        
        # 验证初始条件
        T_initial, rho_initial = model(x_initial)
        T_error_initial = torch.mean(torch.abs(T_initial - 290.0)).item()
        print(f"初始温度误差: {T_error_initial:.4f} K")
        
        # 验证体积约束
        x_space = x_val[:, 0:2]
        rho = model.topology_nn(x_space)
        volume = torch.mean(rho).item()
        volume_min = rho.min().item()
        volume_max = rho.max().item()
        volume_std = rho.std().item()
        print(f"\n体积分数统计:")
        print(f"  平均值: {volume:.4f} (目标: {phi_total})")
        print(f"  范围: [{volume_min:.3f}, {volume_max:.3f}]")
        print(f"  标准差: {volume_std:.4f}")
        
        # 二元化程度
        binary_threshold = 0.5
        binary_ratio = torch.mean((rho > binary_threshold).float()).item()
        print(f"  二元化比例 (> {binary_threshold}): {binary_ratio:.3f}")
        
        # 验证物理合理性
        print(f"\n物理量范围:")
        print(f"  温度范围: {T_initial.min().item():.1f}K - {T_initial.max().item():.1f}K")
    
    # 计算PDE残差
    print("\n计算PDE残差...")
    x_val.requires_grad_(True)
    residuals = model.compute_pde_residuals(x_val)
    
    print("PDE残差:")
    print(f"  能量方程: {torch.mean(residuals['energy']**2).item():.4e}")
    
    # 相变利用情况
    T = residuals['T']
    in_phase_change = torch.sum((T > Tpc - DeltaT/2) & (T < Tpc + DeltaT/2)).item()
    phase_change_ratio = in_phase_change / len(T)
    print(f"\n相变利用情况:")
    print(f"  相变区间占比: {phase_change_ratio:.3f}")
    
    print("="*60)
    
    return residuals

# ==================== 可视化函数 ====================
def visualize_results(model, save_dir):
    """可视化结果"""
    os.makedirs(save_dir, exist_ok=True)
    
    sampler = Sampler(r0, r1)
    
    # 创建均匀网格用于可视化
    grid_points, X, Y, T_grid = sampler.create_grid(N_r=30, N_theta=50, N_t=5)
    grid_points = grid_points.to(device)
    
    # 不同时间点的可视化
    time_indices = [0, 2, 4]  # 选择三个时间点
    
    for t_idx in time_indices:
        # 提取当前时间点的数据
        t_value = T_grid[0, 0, t_idx]
        start_idx = t_idx * (30 * 50)
        end_idx = (t_idx + 1) * (30 * 50)
        
        current_points = grid_points[start_idx:end_idx]
        
        with torch.no_grad():
            T_vals, rho_vals = model(current_points)
        
        # 重塑为网格
        T_grid_vals = T_vals.cpu().numpy().reshape(30, 50)
        rho_grid_vals = rho_vals.cpu().numpy().reshape(30, 50)
        
        # 转换为直角坐标
        X_2d = X[:, :, t_idx]
        Y_2d = Y[:, :, t_idx]
        
        # 绘图
        fig, axes = plt.subplots(2, 2, figsize=(14, 12))
        
        # 温度场
        im1 = axes[0, 0].contourf(X_2d, Y_2d, T_grid_vals, levels=50, cmap='jet')
        axes[0, 0].set_title(f'Temperature Distribution (t={t_value:.2f})')
        axes[0, 0].set_xlabel('x (m)')
        axes[0, 0].set_ylabel('y (m)')
        axes[0, 0].axis('equal')
        plt.colorbar(im1, ax=axes[0, 0])
        
        # 拓扑结构
        im2 = axes[0, 1].contourf(X_2d, Y_2d, rho_grid_vals, levels=50, cmap='binary', vmin=0, vmax=1)
        axes[0, 1].set_title(f'Topology Distribution (t={t_value:.2f})')
        axes[0, 1].set_xlabel('x (m)')
        axes[0, 1].set_ylabel('y (m)')
        axes[0, 1].axis('equal')
        plt.colorbar(im2, ax=axes[0, 1])
        
        # 温度直方图
        T_flat = T_vals.cpu().numpy().flatten()
        if np.ptp(T_flat) > 0:  # 检查数据范围
            axes[1, 0].hist(T_flat, bins=30, color='lightcoral', edgecolor='black', alpha=0.7)
            axes[1, 0].axvline(x=Tpc, color='green', linestyle='--', linewidth=2, 
                              label=f'Phase Change: {Tpc}K')
            axes[1, 0].axvline(x=Tpc - DeltaT/2, color='blue', linestyle=':', linewidth=1.5)
            axes[1, 0].axvline(x=Tpc + DeltaT/2, color='blue', linestyle=':', linewidth=1.5)
            axes[1, 0].set_xlabel('Temperature (K)')
            axes[1, 0].set_ylabel('Frequency')
            axes[1, 0].set_title(f'Temperature Distribution (t={t_value:.2f})')
            axes[1, 0].legend()
            axes[1, 0].grid(True, alpha=0.3)
        else:
            axes[1, 0].text(0.5, 0.5, 'Temperature data is constant', 
                           horizontalalignment='center', verticalalignment='center',
                           transform=axes[1, 0].transAxes, fontsize=12)
            axes[1, 0].set_title('Temperature Distribution')
        
        # 拓扑结构直方图
        rho_flat = rho_vals.cpu().numpy().flatten()
        axes[1, 1].hist(rho_flat, bins=30, color='skyblue', edgecolor='black', alpha=0.7)
        axes[1, 1].axvline(x=phi_total, color='red', linestyle='--', linewidth=2, 
                          label=f'Target: {phi_total}')
        axes[1, 1].set_xlabel('Topology Value')
        axes[1, 1].set_ylabel('Frequency')
        axes[1, 1].set_title(f'Topology Value Distribution (t={t_value:.2f})')
        axes[1, 1].legend()
        axes[1, 1].grid(True, alpha=0.3)
        
        plt.suptitle(f'Results at t={t_value:.2f}', fontsize=16, y=1.02)
        plt.tight_layout()
        plt.savefig(f"{save_dir}/results_t{t_value:.2f}.png", dpi=300, bbox_inches='tight')
        plt.close()
    
    print(f"可视化结果已保存到: {save_dir}")

# ==================== 主程序 ====================
if __name__ == "__main__":
    print("拓扑优化PINN - 相变储热系统（简化稳定版）")
    print("="*60)
    
    # 1. 初始化模型
    print("初始化模型...")
    model = SimplePINN().to(device)
    
    # 设置储热过程
    model.set_heating_mode(True)
    
    # 2. 验证初始模型
    print("验证初始模型...")
    validate_model(model)
    
    # 3. 训练模型
    print("\n开始训练...")
    trainer = Trainer(model)
    trainer.train()
    
    # 4. 验证训练后模型
    print("\n验证训练后模型...")
    residuals = validate_model(model)
    
    # 5. 可视化结果
    print("\n生成可视化结果...")
    visualize_results(model, save_dir=f"{trainer.save_dir}/visualization")
    
    # 6. 分析结果
    print("\n分析优化结果...")
    with torch.no_grad():
        # 采样分析点
        sampler = Sampler(r0, r1)
        x_analysis = sampler.sample_domain(1000).to(device)
        T, rho = model(x_analysis)
        
        # 计算性能指标
        T_avg = torch.mean(T).item()
        T_std = torch.std(T).item()
        volume_actual = torch.mean(rho).item()
        
        print(f"\n性能指标:")
        print(f"  平均温度: {T_avg:.2f} K")
        print(f"  温度标准差: {T_std:.2f} K")
        print(f"  实际体积分数: {volume_actual:.4f}")
        print(f"  目标体积分数: {phi_total}")
        print(f"  体积误差: {abs(volume_actual - phi_total):.4f}")
        
        # 相变利用情况
        in_phase_change = torch.sum((T > Tpc - DeltaT/2) & (T < Tpc + DeltaT/2)).item()
        phase_change_ratio = in_phase_change / len(T)
        print(f"  相变区间占比: {phase_change_ratio:.3f}")
        
        # 二元化设计质量
        binary_score = torch.mean((rho > 0.7).float() + (rho < 0.3).float()).item() / 2
        print(f"  二元化得分: {binary_score:.3f} (1.0为完全二元化)")
    
    print("\n" + "="*60)
    print("程序执行完成!")
    print(f"结果保存目录: {trainer.save_dir}")
    print("="*60)

使用设备: cuda
拓扑优化PINN - 相变储热系统（简化稳定版）
初始化模型...
验证初始模型...

模型验证
内壁温度误差: 35.0001 K
初始温度误差: 35.0000 K

体积分数统计:
  平均值: 0.5000 (目标: 0.3)
  范围: [0.500, 0.500]
  标准差: 0.0002
  二元化比例 (> 0.5): 0.520

物理量范围:
  温度范围: 325.0K - 325.0K

计算PDE残差...
PDE残差:
  能量方程: 6.7920e-08

相变利用情况:
  相变区间占比: 0.000

开始训练...
开始训练简化版拓扑优化PINN

阶段 1/2: 预训练
轮次: 100, 学习率: 0.001
Epoch   10 | Loss: 1.3098e+05 | Volume: 0.4750 | PDE: 2.14e-02 | BC: 1.18e+03 | LR: 9.78e-04
Epoch   20 | Loss: 1.1531e+05 | Volume: 0.4358 | PDE: 1.97e+00 | BC: 1.01e+03 | LR: 9.14e-04
Epoch   30 | Loss: 7.6546e+04 | Volume: 0.3830 | PDE: 2.90e+01 | BC: 5.71e+02 | LR: 8.15e-04
Epoch   40 | Loss: 4.5912e+04 | Volume: 0.3266 | PDE: 6.49e+01 | BC: 1.57e+02 | LR: 6.89e-04
Epoch   50 | Loss: 4.2578e+04 | Volume: 0.2873 | PDE: 4.68e+01 | BC: 3.79e+01 | LR: 5.50e-04
Epoch   60 | Loss: 4.1304e+04 | Volume: 0.3036 | PDE: 7.33e+01 | BC: 4.09e+01 | LR: 4.11e-04
Epoch   70 | Loss: 4.0926e+04 | Volume: 0.3017 | PDE: 1.11e+02 | BC: 6.93e+01 | LR: 2.85e-04
Epoch   